<a href="https://colab.research.google.com/github/doubletran/resnet14-cifar10-pytorch/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


## Datasets and Dataloaders

In [ ]:
transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),torchvision.transforms.Normalize((0.1307,), (0.3081,))])

train_dataset = torchvision.datasets.MNIST('/files/', train=True, download=True, transform=transform)
val_dataset = torchvision.datasets.MNIST('/files/', train=False, download=True, transform=transform)

trainloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valloader = DataLoader(val_dataset, batch_size=128, shuffle=False)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.04MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.9MB/s]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualizeBatch(x_batch,pred, num=10):
  f, axs = plt.subplots(num, 2, figsize=(14,6*num))
  for i in range(0,num):
    axs[i,0].imshow(x_batch[i,:,:].squeeze(), cmap='gray')
    axs[i,1].bar(list(range(10)),pred[i,:], label=list(range(10)))

  return f


## Layers and Models

In [ ]:
class ResBlock(nn.Module):
    def __init__(self,inplanes: int,planes: int, stride: int = 1, padding:int = 1) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(inplanes,planes,kernel_size=3,
                              stride=stride,padding=padding,bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        #self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes,planes,kernel_size=3,
                              stride=stride,padding=padding,bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)
        #out = self.relu1(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu2(out)

        return out

In [ ]:
class MyResNet(nn.Module):

  def __init__(self,inplanes: int, classes: int,) -> None:
    super().__init__()
    self.conv1 = nn.Conv2d(1,128,kernel_size=3, padding=1, bias=True)
    self.block1 = ResBlock(128,128)
    self.block2 = ResBlock(128,128)
    self.conv1x1 = nn.Conv2d(128,classes,kernel_size=1, bias=True)
    self.globalpool = nn.AdaptiveMaxPool2d((1,1))

  def forward(self, x: Tensor) -> Tensor:
    out = self.conv1(x)
    out = F.relu(out)
    out = self.block1(out)
    out = self.block2(out)
    out = self.conv1x1(out)
    out = self.globalpool(out)
    return out.squeeze()

In [ ]:
model = MyResNet(3, 10)
print(model)

MyResNet(
  (conv1): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (block1): ResBlock(
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu2): ReLU(inplace=True)
  )
  (block2): ResBlock(
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu2): ReLU(inplace=True)
  )
  (conv1x1): Conv2d(128, 10, kernel_size=(1, 1), stride=(

## Training

In [ ]:
import wandb

wandb.login()


device = torch.device('cuda')

model.to(device)


config = {'epochs': 50, 'lr': 3e-3}
iter = 0
with wandb.init(config = config) as run:

    wandb.define_metric("train/iter_loss", step_metric="global_step")
    wandb.define_metric("val/epoch_loss", step_metric="epoch")
    wandb.define_metric("val/epoch_accuracy", step_metric="epoch")

    optimizer = torch.optim.SGD(model.parameters(), lr=run.config['lr'], weight_decay=0.0)
    criterion = torch.nn.CrossEntropyLoss()
    for i in range(run.config['epochs']):
        model.train()
        print("Epoch {}".format(i))
        for j,input in enumerate(trainloader,0):
            iter+=1



            x = input[0].to(device)
            y = input[1].to(device)

            if iter == 0:
                image = wandb.Image(x[0,:,:,:])
                run.log({"example": image})

            out = model(x)
            loss = criterion(out,y)
            model.conv1.weight.requires_grad = False

            model.zero_grad()
            loss.backward()

            optimizer.step()

            _, predicted = torch.max(out.data, 1)
            correct = (predicted == y).float().mean().item()

            run.log({"train/train_loss": loss.item(), "train/train_accuracy": correct}, step = iter)



        model.eval()
        running_loss = 0
        running_acc = 0
        for j,input in enumerate(valloader,0):

             x = input[0].to(device)
             y = input[1].to(device)

             out = model(x)
             loss = criterion(out,y)

             _, predicted = torch.max(out.data, 1)
             correct = (predicted == y).sum().item()

             running_loss += loss.item()
             running_acc += correct

             if j == 0:
                  f = visualizeBatch(x.detach().cpu(),F.softmax(out,dim=1).detach().cpu())
                  run.log({"batch0": f})
                  plt.close(f)
        run.log({"epoch": i, "val/epoch_loss": running_loss / len(valloader), "val/epoch_accuracy": running_acc / len(val_dataset)}, step = iter + 1)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:



KeyboardInterrupt: 